In [1]:
from pyspark.sql import SparkSession
import os

# Variáveis essenciais
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_SESSION_TOKEN = os.getenv("AWS_SESSION_TOKEN")
S3_BUCKET = os.getenv("S3_BUCKET")
S3_PATH = f"s3a://{S3_BUCKET}/_conexao_teste/teste"

spark = (
    SparkSession.builder
    .appName("pipeline-alfabetizacao")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.TemporaryAWSCredentialsProvider")
    .config("spark.hadoop.fs.s3a.access.key", AWS_ACCESS_KEY_ID)
    .config("spark.hadoop.fs.s3a.secret.key", AWS_SECRET_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.session.token", AWS_SESSION_TOKEN)
    .config("spark.hadoop.fs.s3a.endpoint", "s3.amazonaws.com")
    .getOrCreate()
)

26/07/04 08:52:56 WARN Utils: Your hostname, lua resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/07/04 08:52:56 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/diego/miniconda3/envs/postech2/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/diego/.ivy2/cache
The jars for the packages stored in: /home/diego/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-184068c7-46ed-4035-bfd9-905bcc8e134e;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 296ms :: artifacts dl 13ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evict

In [2]:
# Teste de conexão
## Criar dados de teste
data = [("Teste de Conexao", 1)]
df = spark.createDataFrame(data, ["parametro", "valor"])

## Escrita (Overwrite para evitar erro de arquivo existente)
df.write.mode("overwrite").parquet(S3_PATH)

# 3. Leitura e Verificação
df = spark.read.parquet(S3_PATH)
df.show()

26/07/04 08:53:47 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


+----------------+-----+
|       parametro|valor|
+----------------+-----+
|Teste de Conexao|    1|
+----------------+-----+

